In [1]:
from fastapi import FastAPI, Query, HTTPException, Header, Response
from fastapi.responses import JSONResponse, StreamingResponse
import requests
import xmltodict
import pandas as pd
from io import StringIO

# Parameters

SOURCES = {
    "B01":"bionmass",
    "B02":"brown_coal_lignite",
    "B03":"coal",
    "B04":"gas",
    "B05":"hard_coal",
    "B06":"fossil_oil",
    "B07":"fossil_oil_shale",
    "B08":"fossil_peat",
    "B09":"geothermal",
    "B10":"hydro_pumped_storage",
    "B11":"hydro_run_of_river_and_poundage",
    "B12":"hydro_water_reservoir",
    "B13":"marine",
    "B14":"nuclear",
    "B15":"other_renewable",
    "B16":"solar",
    "B17":"waste",
    "B18":"wind_offshore",
    "B19":"wind_onshore",
    "B20":"other",
    "B25":"energy_storage"

}

MONTHS = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]
DAYS_IN_MONTH = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

import re

def parse_resolution(res_str):
    # Example: PT15M or PT1H
    match = re.match(r"PT(\d+)([HMS])", res_str)
    if not match:
        return 60  # default to 60 mins
    value, unit = match.groups()
    value = int(value)
    if unit == "H":
        return value * 60
    elif unit == "M":
        return value
    elif unit == "S":
        return value / 60
    return 60

# Functions
def get_eic_code(country_code=None, country_name=None):
    import pandas as pd
    df = pd.read_csv("data/energy_mix/raw/eu_api/eic_codes.csv")
    if country_code is not None:
        result = df[df["CountryCode"].str.strip().str.upper() == country_code.strip().upper()]
        if not result.empty:
            return result.iloc[0]["Code"]
    if country_name is not None:
        result = df[df["CountryName"].str.strip().str.upper() == country_name.strip().upper()]
        if not result.empty:
            return result.iloc[0]["Code"]
    if country_code is None and country_name is None:
        #raise ValueError("Either country_code or country_name must be provided")
        raise HTTPException(status_code=400, detail="Either country_code or country_name must be provided")
    else: 
        #raise ValueError("No matching EIC code found for the provided country code or name")
        raise HTTPException(status_code=400, detail="Unsupported country code")

def to_dataframe(xml_content):
    #with open(xml_file, "r") as f:
    #    xml_content = f.read()

    data_dict = xmltodict.parse(xml_content)
    
    time_series = data_dict["GL_MarketDocument"].get("TimeSeries", [])

    if not isinstance(time_series, list):
        time_series = [time_series]

    records = []
    keys = []
        
    for series in time_series:

        prod_type = series["MktPSRType"]["psrType"]

        if "inBiddingZone_Domain.mRID" in series: 
            print(f"Skipping series with inBiddingZone_Domain.mRID {prod_type}")
            continue
        period  = series["Period"]
        resolution = period.get("resolution", "PT60M")
        minutes_per_step = parse_resolution(resolution)
        dt_start = period["timeInterval"]["start"]
        
        key = (dt_start, SOURCES[prod_type])
        if key in keys:
            print(f"Skipping duplicate entry for {key}")
            continue
        keys.append(key)
        
        points = period.get("Point", [])
        if not isinstance(points, list):
            points = [points]
        for p in points:
            position = int(p["position"]) - 1
            quantity = float(p["quantity"])
            dt = pd.to_datetime(dt_start) + pd.to_timedelta(position * minutes_per_step, unit="min")
            records.append({
                #"position": position,
                "timestamp": dt,
                "type": SOURCES[prod_type],
                "MW": quantity
            })
          
    
    df = pd.DataFrame(records)
    
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.pivot(index="timestamp", columns="type", values="MW").fillna(0).reset_index()    
    df["timestamp"] = df["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    return df
    
def to_mix_dataframe(df):
    df_std = pd.DataFrame(columns=["timestamp", "nuclear", "geothermal", "biomass", "coal", "wind", "solar", "hydro", "gas", "oil", "unknown"])
    df_std["timestamp"] = df["timestamp"]
    df_std["nuclear"] = df.get("nuclear", 0)
    df_std["geothermal"] = df.get("geothermal", 0)
    df_std["biomass"] = df.get("bionmass", 0)
    df_std["coal"] = df.get("coal", 0) + df.get("brown_coal_lignite", 0) + df.get("hard_coal", 0)
    df_std["wind"] = df.get("wind_offshore", 0) + df.get("wind_onshore", 0)
    df_std["solar"] = df.get("solar", 0)
    df_std["hydro"] = df.get("hydro_pumped_storage", 0) + df.get("hydro_run_of_river_and_poundage", 0) + df.get("hydro_water_reservoir", 0)
    df_std["gas"] = df.get("gas", 0)
    df_std["oil"] = df.get("fossil_oil", 0) + df.get("fossil_oil_shale", 0) + df.get("fossil_peat", 0)
    df_std["unknown"] = df.get("other", 0) + df.get("other_renewable", 0) + df.get("marine", 0) + df.get("waste", 0) + df.get("energy_storage", 0)

    df_std = df_std.fillna(0)
    normalize_df = lambda df: df.div(df.sum(axis=1), axis=0)
    df_std.set_index("timestamp", inplace=True)
    df_std = normalize_df(df_std)

    return df_std

def sanity_check(country, start, end):
    eic_code = get_eic_code(country_code=country)
    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)
    if start_dt > end_dt:
        raise HTTPException(status_code=400, detail="Start date must be before end date")
    return eic_code
    
def get_chunks(start, end):

    start_year = int(start[:4])
    end_year = int(end[:4])
    start_month = int(start[4:6])
    end_month = int(end[4:6])

    chunks = []
    for year in range(start_year, end_year + 1):
        # Determine the first and last month for this year
        if year == start_year:
            m_start = start_month
        else:
            m_start = 1
        if year == end_year:
            m_end = end_month
        else:
            m_end = 12

        for m in range(m_start, m_end + 1):
            month_str = f"{m:02d}"
            year_str = str(year)
            if year == start_year and m == start_month:
                chunk_start = start
            else:
                chunk_start = f"{year_str}{month_str}010000"
            if year == end_year and m == end_month:
                chunk_end = end
            else:
                days = DAYS_IN_MONTH[m - 1]
                chunk_end = f"{year_str}{month_str}{days:02d}0000"
            chunks.append((chunk_start, chunk_end))
    return chunks



app = FastAPI(title="ENTSO-E Proxy API (BYOK)")

ENTSOE_API_BASE = "https://web-api.tp.entsoe.eu/api"

@app.get("/generation")
def get_generation(
    country: str = Query(..., description="Country code, e.g., IT, DE, FR"),
    start: str = Query(..., description="Start date YYYYMMDDHHMM"),
    end: str = Query(..., description="End date YYYYMMDDHHMM"),
    token: str = Header(..., description="Your ENTSO-E API key")
):
    
    eic_code = sanity_check(country, start, end)
    chunks = get_chunks(start, end)

    mix_df = pd.DataFrame()

    for chunk_start, chunk_end in chunks:
        params = {
            "securityToken": token,     # securityToken={API_KEY}
            "documentType": "A75",      # documentType=A75
            "processType": "A16",       # processType=A16
            "in_Domain": eic_code,      # in_Domain={EIC_CODE}
            "periodStart": chunk_start, # periodStart={start}
            "periodEnd": chunk_end      # periodEnd={end}
        }

        res = requests.get(ENTSOE_API_BASE, params=params)

        if res.status_code != 200:
            raise HTTPException(status_code=500, detail=f"Error from ENTSO-E API\n{res}")

        try:
            df = to_dataframe(res.text)
            print(f"Processing chunk: {chunk_start} to {chunk_end}")
            mix_df = pd.concat([mix_df, to_mix_dataframe(df)])
            mix_df.drop_duplicates(inplace=True)
            print(mix_df.head())
            
        except Exception as e:
            print(f"Failed to parse ENTSO-E response for chunk {chunk_start} to {chunk_end}: {e}")
            output = StringIO()
            mix_df.to_csv(output)
            print(mix_df.head())
            output.seek(0)
            return StreamingResponse(output, media_type="text/csv", headers={"Content-Disposition": f"attachment;"})
            #raise HTTPException(status_code=500, detail=f"Failed to parse ENTSO-E response: {e}")

    print(f"Final DataFrame shape: {mix_df.shape}")
    output = StringIO()
    mix_df.to_csv(output)
    print(mix_df.head())
    output.seek(0)
    return StreamingResponse(output, media_type="text/csv", headers={"Content-Disposition": f"attachment;"})
            



In [2]:
country= "IE"
start = "202405200000"
end = "202406010100"
token = "7ef1c19d-b6dd-465b-9d5b-99e1e4d41400"

In [3]:
get_generation(
    country=country,
    start=start,
    end=end,
    token="7ef1c19d-b6dd-465b-9d5b-99e1e4d41400")

Skipping series with inBiddingZone_Domain.mRID B04
Skipping series with inBiddingZone_Domain.mRID B04
Skipping series with inBiddingZone_Domain.mRID B05
Skipping series with inBiddingZone_Domain.mRID B05
Skipping series with inBiddingZone_Domain.mRID B06
Skipping series with inBiddingZone_Domain.mRID B06
Skipping series with inBiddingZone_Domain.mRID B08
Skipping series with inBiddingZone_Domain.mRID B08
Skipping series with inBiddingZone_Domain.mRID B10
Skipping series with inBiddingZone_Domain.mRID B10
Skipping series with inBiddingZone_Domain.mRID B11
Skipping series with inBiddingZone_Domain.mRID B11
Skipping series with inBiddingZone_Domain.mRID B20
Skipping series with inBiddingZone_Domain.mRID B20
Skipping series with inBiddingZone_Domain.mRID B19
Skipping series with inBiddingZone_Domain.mRID B19
Processing chunk: 202405200000 to 202405310000
                      nuclear  geothermal  biomass      coal  wind  solar  \
timestamp                                                   

In [9]:
! curl -H "token: 7ef1c19d-b6dd-465b-9d5b-99e1e4d41400" \
  "http://localhost:8000/generation?country=IE&start=202411130830&end=202501010000" \
  -o data/energy_mix/raw/ie_nov_dic.csv


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   319    0   319    0     0    114      0 --:--:--  0:00:02 --:--:--   1140      0 --:--:--  0:00:02 --:--:--     0


In [ ]:
import matplotlib.pyplot as plt

for key in df:
    grouped = df[key].groupby(["timestamp", "type"], as_index=False).count()
    pivot = grouped.pivot(index="timestamp", columns="type", values="MW").fillna(0)
    pivot.plot(kind="bar", stacked=True, figsize=(15, 6))
    plt.title(f"Counts by timestamp and type for chunk {key}")
    plt.ylabel("Count")
    plt.xlabel("Timestamp")
    plt.tight_layout()
    plt.show()


    duplicates = grouped[grouped["MW"] > 1][["timestamp", "type", "MW"]]
    print(duplicates)

In [ ]:
import os
df = {}

In [ ]:
eic_code = sanity_check(country, start, end)
chunks = get_chunks(start, end)



for chunk_start, chunk_end in chunks:
    params = {
        "securityToken": token,     # securityToken={API_KEY}
        "documentType": "A75",      # documentType=A75
        "processType": "A16",       # processType=A16
        "in_Domain": eic_code,      # in_Domain={EIC_CODE}
        "periodStart": chunk_start, # periodStart={start}
        "periodEnd": chunk_end      # periodEnd={end}
    }
    
    if os.path.exists(f"data/energy_mix/raw/eu_api/{country}_{chunk_start}_{chunk_end}.xml"):
        with open(f"data/energy_mix/raw/eu_api/{country}_{chunk_start}_{chunk_end}.xml", "r") as f:
            xml_content = f.read()
    else:
        res = requests.get(ENTSOE_API_BASE, params=params)

        if res.status_code != 200:
            raise HTTPException(status_code=500, detail=f"Error from ENTSO-E API\n{res}")

        xml_content = res.text
        with open(f"data/energy_mix/raw/eu_api/{country}_{chunk_start}_{chunk_end}.xml", "w") as f:
            f.write(xml_content)
    df[chunk_start] = to_dataframe(xml_content)


In [ ]:
df

In [ ]:
df['202405200000'].pivot(index="timestamp", columns="type", values="MW").fillna(0).reset_index()    